In [23]:
import torch 
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import matplotlib.pyplot as plt

In [24]:
torch.manual_seed(100)

In [25]:
import pandas as pd

train_data = pd.read_csv("fashion_train.csv")
test_data = pd.read_csv("fashion_test.csv")

In [26]:
train_data.columns

Index(['label', 'pixel1', 'pixel2', 'pixel3', 'pixel4', 'pixel5', 'pixel6',
       'pixel7', 'pixel8', 'pixel9',
       ...
       'pixel775', 'pixel776', 'pixel777', 'pixel778', 'pixel779', 'pixel780',
       'pixel781', 'pixel782', 'pixel783', 'pixel784'],
      dtype='str', length=785)

In [27]:
print("test:" ,test_data.shape)
print("train:" ,train_data.shape)

test: (10000, 785)
train: (60000, 785)


In [28]:
X_train = train_data.drop("label", axis=1)
y_train = train_data["label"]

X_test = test_data.drop("label", axis=1)
y_test = test_data["label"]


In [29]:
X_train = X_train/255.0
X_test = X_test/255.0

In [30]:
X_train = X_train.to_numpy()
y_train = y_train.to_numpy()

X_test = X_test.to_numpy()
y_test = y_test.to_numpy()

In [31]:
class CustomDataset(Dataset) :
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.int64)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index] , self.labels[index]

In [32]:
train_dataset = CustomDataset(X_train,y_train)
test_dataset = CustomDataset(X_test,y_test)
print(len(train_dataset))
print(len(test_dataset))

60000
10000


In [ ]:
train_loader = DataLoader(train_dataset , batch_size = 32 , shuffle=True)
test_loader = DataLoader(test_dataset , batch_size = 32 , shuffle=False)

In [35]:
class NN(nn.Module):
    def __init__(self,input):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    def forward(self, x):
        return self.model(x)

    

In [43]:
epochs = 200
lr = 0.01

In [44]:
model = NN(X_train.shape[1])

loss_function = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters() , lr=lr)

In [45]:
print("batches:" , len(train_loader))

batches: 1875


Epoch 1 → see all 60,000 images
Epoch 2 → see all 60,000 images
...

in one epoch:
60k images -> 1875 batches -> 1875 weight updtaes 

in 10 epochs : 1875 * 10 = 18750 updates


In [46]:
for epoch in range(epochs):

    total_loss = 0

    for batch_features , batch_labels in train_loader:

        out = model(batch_features)

        loss = loss_function(out,batch_labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"epoch {epoch + 1}/{epochs}, loss: {avg_loss}")



epoch 1/200, loss: 1.095111342604955
epoch 2/200, loss: 0.5902716468811036
epoch 3/200, loss: 0.5116340886116028
epoch 4/200, loss: 0.4760771429379781
epoch 5/200, loss: 0.45325151134729386
epoch 6/200, loss: 0.4330390731374423
epoch 7/200, loss: 0.41792521862983706
epoch 8/200, loss: 0.4053242380976677
epoch 9/200, loss: 0.39182748641173043
epoch 10/200, loss: 0.38273131258090337
epoch 11/200, loss: 0.3726873764077822
epoch 12/200, loss: 0.36402340845664344
epoch 13/200, loss: 0.35662541067997616
epoch 14/200, loss: 0.3494744809269905
epoch 15/200, loss: 0.3413420944492022
epoch 16/200, loss: 0.33606124681830407
epoch 17/200, loss: 0.3290784635404746
epoch 18/200, loss: 0.32391087408065794
epoch 19/200, loss: 0.3177877643962701
epoch 20/200, loss: 0.3129799776593844
epoch 21/200, loss: 0.30755687348445254
epoch 22/200, loss: 0.3025940576871236
epoch 23/200, loss: 0.2979540295680364
epoch 24/200, loss: 0.2938151848634084
epoch 25/200, loss: 0.289431346676747
epoch 26/200, loss: 0.28478

In [47]:
model.eval()

NN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [48]:

total = 0
correct = 0

with torch.no_grad():
    for batch_features, batch_labels in test_loader:
        outputs = model(batch_features)
        
        _, predicted = torch.max(outputs, 1)
        
        total = total + batch_labels.shape[0]
        
        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.8849
